In [1]:
import xarray as xr, netCDF4 as nc, numpy as np, pandas as pd, os, datetime

from scipy.spatial import cKDTree

import matplotlib.pyplot as plt
import plotly.graph_objects as go

In [2]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')

ehf_fpath = '/scratch/ng72/ms5578'
nmap_path = '/g/data/ng72/ms5578/ID_HW_BARRA/data/raw'
write_path = '/scratch/ng72/ms5578/time_series'
raw_NEM_cache = '/scratch/ng72/ms5578/nemosis_cache'

In [3]:
sdate, edate = "2018/10/01 00:00:00", "2019/03/31 23:59:59"

In [4]:
yr_file = f"{ehf_fpath}/hw_files/HW_EHF_2018_2019.nc"
ds = xr.open_dataset(yr_file,
                         engine='netcdf4',
                         chunks="auto")

EHF_ds = ds.sel(time=slice(sdate, edate))

In [5]:
gen_df = pd.read_csv(f"{nmap_path}/nmap.csv")
gen_df.drop(gen_df.columns[[3, 1, 4, 5]], axis=1, inplace=True)

gen_df.columns = (
    gen_df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace(r'[^\w_]', '', regex=True)
    .str.replace('__', '_')
)

First we join the generator information with the EHF info, sticking to daily intervals

In [6]:
def merge_nearest(ds,df):
    # Extract latitude and longitude grids
    lat_grid = ds['lat'].values
    lon_grid = ds['lon'].values
    
    # Convert gen_df lat/lon to numpy arrays for vectorized operations
    gen_lats = df['lat'].values
    gen_lons = df['lon'].values
    
    # Find the nearest indices for all latitudes and longitudes in gen_df
    lat_indices = np.abs(lat_grid[:, None] - gen_lats).argmin(axis=0)
    lon_indices = np.abs(lon_grid[:, None] - gen_lons).argmin(axis=0)
    
    # Get the nearest latitudes and longitudes
    nearest_lats = lat_grid[lat_indices]
    nearest_lons = lon_grid[lon_indices]
    
    # Create DataArrays for vectorized selection
    nearest_lats_da = xr.DataArray(nearest_lats, dims="DUID")
    nearest_lons_da = xr.DataArray(nearest_lons, dims="DUID")
    
    new_ds = ds.sel(lat=nearest_lats_da, lon=nearest_lons_da)
    new_ds = new_ds.assign_coords(DUID=df['duid'])

    return new_ds

Performing some other operations to tidy up heatwave_info

In [7]:
hw_info = merge_nearest(EHF_ds, gen_df).to_dataframe().reset_index()
hw_info['time'] = pd.to_datetime(hw_info['time'])
hw_info = hw_info.replace(1.000000e+20, np.nan)
hw_info = hw_info.drop(['height','crs'], axis=1)
hw_info = hw_info.sort_values(by=['DUID', 'time'])


Here we number consecutive heatwave days

In [8]:
def number_heatwave_days(group):
    # Identify where heatwave days start (either first row or break in sequence)
    is_heatwave = group['EHF_flag'] == 1
    # Create a group ID for each continuous block of heatwave days
    group['event_group'] = (is_heatwave & (is_heatwave != is_heatwave.shift(1))).cumsum()
    # Set non-heatwave days to NaN in group
    group.loc[~is_heatwave, 'event_group'] = pd.NA
    # Now, within each event_group, number the days
    group['HW_event_day'] = group.groupby('event_group').cumcount() + 1
    # Drop the helper column
    group = group.drop(columns='event_group')
    return group

# Apply by DUID
hw_info = hw_info.groupby('DUID', group_keys=False).apply(number_heatwave_days, include_groups=True)
hw_info = hw_info.drop(columns=['lat','lon'])

/jobfs/141456136.gadi-pbs/ipykernel_2357877/3713384521.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  hw_info = hw_info.groupby('DUID', group_keys=False).apply(number_heatwave_days, include_groups=True)


In [9]:
hw_info.to_csv(f"{write_path}/gen_hw_status.csv",
              index=False)

In [10]:
gen_df.to_csv('/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/nmap.csv',
             index=False)

In [11]:
hw_info[hw_info['DUID'].isin(gen_df['duid'][gen_df['region']=="TAS1"])]

,time,DUID,EHF_flag,HW_EHF_avg,HW_EHF_peak,tas_3d_avg,tas_3d_peak,HW_event_day
28,2018-10-01 12:00:00,BASTYAN,0.0,NaN,NaN,NaN,NaN,NaN
515,2018-10-02 12:00:00,BASTYAN,0.0,NaN,NaN,NaN,NaN,NaN
1002,2018-10-03 12:00:00,BASTYAN,0.0,NaN,NaN,NaN,NaN,NaN
1489,2018-10-04 12:00:00,BASTYAN,0.0,NaN,NaN,NaN,NaN,NaN
1976,2018-10-05 12:00:00,BASTYAN,0.0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
86219,2019-03-27 12:00:00,WOOLNTH1,0.0,NaN,NaN,NaN,NaN,NaN
86706,2019-03-28 12:00:00,WOOLNTH1,0.0,NaN,NaN,NaN,NaN,NaN
87193,2019-03-29 12:00:00,WOOLNTH1,0.0,NaN,NaN,NaN,NaN,NaN
87680,2019-03-30 12:00:00,WOOLNTH1,0.0,NaN,NaN,NaN,NaN,NaN


In [12]:
# output = pd.read_csv(f"{write_path}/generation.csv", index_col=0)

# output = output[output['DUID'].isin(hw_info['DUID'])]
# hw_info = hw_info[hw_info['DUID'].isin(output['DUID'])].set_index('time')

# unit_gen = output['TOTALMWh'][output['DUID'] == 'POAT110']
# unit_hw = hw_info[hw_info['DUID'] == 'POAT110']
# unit_gen.index = pd.to_datetime(unit_gen.index)
# unit_gen = unit_gen.resample('1h',label='right').agg({'TOTALMWh': 'sum'})

# df = pd.merge_asof(unit_gen,
#                    unit_hw,
#                    left_index=True,
#                    right_index=True,
#                    tolerance=pd.Timedelta("1d"),
#                    direction='nearest').reset_index()

# def highLights(df, fig, variable, level, mode, fillcolor, layer):
#     """
#     Set a specified color as background for given
#     levels of a specified variable using a shape.
    
#     Keyword arguments:
#     ==================
#     fig -- plotly figure
#     variable -- column name in a pandas dataframe
#     level -- int or float
#     mode -- set threshold above or below
#     fillcolor -- any color type that plotly can handle
#     layer -- position of shape in plotly fiugre, like "below"
    
#     """
    
#     if mode == 'above':
#         m = df[variable].gt(level)
    
#     if mode == 'below':
#         m = df[variable].lt(level)
        
#     df1 = df[m].groupby((~m).cumsum())['time'].agg(['first','last'])

#     for index, row in df1.iterrows():
#         #print(row['first'], row['last'])
#         fig.add_shape(type="rect",
#                         xref="x",
#                         yref="paper",
#                         x0=row['first'],
#                         y0=0,
#                         x1=row['last'],
#                         y1=1,
#                         line=dict(color="rgba(0,0,0,0)",width=3,),
#                         fillcolor=fillcolor,
#                         layer=layer) 
#     return(fig)

# # Create figure
# fig = go.Figure()

# fig.add_trace(
#     go.Scatter(x=df['time'], y=df['TOTALMWh']))

# # Set title
# fig.update_layout(
#     title_text="Time series with range slider and selectors"
# )

# # Add range slider
# fig.update_layout(
#     xaxis=dict(
#         rangeselector=dict(
#             buttons=list([
#                 dict(count=7,
#                      label="1w",
#                      step="day",
#                      stepmode="backward"),
#                 dict(count=1,
#                      label="1m",
#                      step="month",
#                      stepmode="backward"),
#                 dict(step="all")
#             ])
#         ),
#         rangeslider=dict(
#             visible=True
#         ),
#         type="date"
#     )
# )

# fig = highLights(df=df, fig = fig, variable = 'EHF_flag', level = 0, mode = 'above',
#                fillcolor = 'rgba(200,0,200,0.2)', layer = 'below')

# fig.show()